# Model Setup

In [1]:
%load_ext autoreload
%autoreload 2

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import json
import sys
import time

import torch
import pandas as pd  # requires: pip install 'pandas[pyarrow]'
from chronos import Chronos2Pipeline

from chronos import BaseChronosPipeline

pipeline2 = BaseChronosPipeline.from_pretrained("amazon/chronos-bolt-base", device_map="mps")


os.chdir("*/Setup/1. Environment/")

from data_generator import make_walk
from dataset import build_dataset, load_overview, load_window
from metrics import conditional_mean_forecaster, naive_forecaster, squared_error, ratio_rmse_floor, skill_vs_naive, skill_vs_conditional, drift_mle, plugin_forecaster


Loading weights:   0%|          | 0/269 [00:00<?, ?it/s]

In [2]:
def forecast_chronos_bolt(context, horizon, pipeline):
    context = np.asarray(context, dtype=np.float32)

    with torch.no_grad():
        forecast = pipeline.predict(torch.tensor(context), horizon)

    quantiles = np.asarray(forecast[0].cpu())      # (num_quantiles, horizon)
    prediction = quantiles[quantiles.shape[0] // 2]

    if len(prediction) != horizon:
        raise ValueError(f"got {len(prediction)}, expected {horizon}")

    return prediction

In [3]:
data_path = "*/"

In [4]:

overview_data = load_overview(data_path)

ORIGIN      = int(overview_data["origin"].iloc[0])
MAX_HORIZON = int(overview_data["max_horizon"].iloc[0])
HORIZONS    = range(1, MAX_HORIZON + 1)
SIGMA = float(overview_data["sigma"].iloc[0])
print(ORIGIN, MAX_HORIZON, SIGMA, len(overview_data))

512 256 1.0 12500


In [5]:
overview_data

,walk_id,snr,seed,sigma,mu,origin,max_horizon
0,snr0.01_seed0,0.01,0,1.0,0.01,512,256
1,snr0.01_seed1,0.01,1,1.0,0.01,512,256
2,snr0.01_seed2,0.01,2,1.0,0.01,512,256
3,snr0.01_seed3,0.01,3,1.0,0.01,512,256
4,snr0.01_seed4,0.01,4,1.0,0.01,512,256
...,...,...,...,...,...,...,...
12495,snr5_seed95,5.00,95,1.0,5.00,512,256
12496,snr5_seed96,5.00,96,1.0,5.00,512,256
12497,snr5_seed97,5.00,97,1.0,5.00,512,256
12498,snr5_seed98,5.00,98,1.0,5.00,512,256


In [6]:
import logging
logging.getLogger("chronos").setLevel(logging.ERROR)

In [7]:
import warnings
warnings.filterwarnings("ignore", message=".*prediction length.*")

In [8]:
import warnings
import logging

warnings.filterwarnings("ignore")

logging.getLogger("chronos").setLevel(logging.ERROR)
logging.getLogger().setLevel(logging.ERROR)

# Implementation

## L = 512

In [9]:
CONTEXT_LEN = 512

In [10]:
rows_context_512 = []

for i in range(len(overview_data)):
    walk = overview_data.iloc[i]

    context, future = load_window(
        walk["walk_id"], CONTEXT_LEN, ORIGIN, MAX_HORIZON, data_path
    )

    if i % 500 == 0:
        print(f"\r{i}/{len(overview_data)}", end="", flush=True)

    prediction  = forecast_chronos_bolt(context, MAX_HORIZON, pipeline2)
    plugin      = plugin_forecaster(context, MAX_HORIZON)
    conditional = conditional_mean_forecaster(context, MAX_HORIZON, walk["mu"])
    naive       = naive_forecaster(context, MAX_HORIZON)

    for h in HORIZONS:
        rows_context_512.append({
            "walk_id":     walk["walk_id"],
            "snr":         walk["snr"],
            "seed":        walk["seed"],
            "sigma":       walk["sigma"],
            "mu":          walk["mu"],
            "context_len": CONTEXT_LEN,
            "horizon":     h,
            "model":       "chronos-2",
            "checkpoint":  "chronos-2-amazon",
            "num_samples": 100,
            "actual":      future[h - 1],
            "forecast":    prediction[h - 1],
            "naive":       naive[h - 1],
            "plugin":      plugin[h - 1],
            "conditional": conditional[h - 1],
            "se_model":       squared_error(prediction,  future, h),
            "se_naive":       squared_error(naive,       future, h),
            "se_plugin":      squared_error(plugin,      future, h),
            "se_conditional": squared_error(conditional, future, h),
        })

predictions_context_512 = pd.DataFrame(rows_context_512)

12000/12500

In [11]:
predictions_context_512

,walk_id,snr,seed,sigma,mu,context_len,horizon,model,checkpoint,num_samples,actual,forecast,naive,plugin,conditional,se_model,se_naive,se_plugin,se_conditional
0,snr0.01_seed0,0.01,0,1.0,0.01,512,1,chronos-2,chronos-2-amazon,100,0.152914,-0.281296,0.0,-0.014330,0.01,0.188538,2.338258e-02,0.027970,0.020424
1,snr0.01_seed0,0.01,0,1.0,0.01,512,2,chronos-2,chronos-2-amazon,100,-0.381044,-0.297376,0.0,-0.028660,0.02,0.007000,1.451946e-01,0.124175,0.160836
2,snr0.01_seed0,0.01,0,1.0,0.01,512,3,chronos-2,chronos-2-amazon,100,-0.504496,-0.273522,0.0,-0.042990,0.03,0.053349,2.545158e-01,0.212988,0.285686
3,snr0.01_seed0,0.01,0,1.0,0.01,512,4,chronos-2,chronos-2-amazon,100,0.803376,-0.231521,0.0,-0.057320,0.04,1.071012,6.454132e-01,0.740797,0.582743
4,snr0.01_seed0,0.01,0,1.0,0.01,512,5,chronos-2,chronos-2-amazon,100,-0.154444,-0.089266,0.0,-0.071649,0.05,0.004248,2.385287e-02,0.006855,0.041797
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3199995,snr5_seed99,5.00,99,1.0,5.00,512,252,chronos-2,chronos-2-amazon,100,1260.902970,979.484192,0.0,1265.948489,1260.00,79196.528390,1.589876e+06,25.457272,0.815354
3199996,snr5_seed99,5.00,99,1.0,5.00,512,253,chronos-2,chronos-2-amazon,100,1263.939192,984.649353,0.0,1270.972095,1265.00,78002.814203,1.597542e+06,49.461718,1.125313
3199997,snr5_seed99,5.00,99,1.0,5.00,512,254,chronos-2,chronos-2-amazon,100,1268.951206,989.878845,0.0,1275.995700,1270.00,77881.382576,1.610237e+06,49.624891,1.099969
3199998,snr5_seed99,5.00,99,1.0,5.00,512,255,chronos-2,chronos-2-amazon,100,1273.053372,982.511658,0.0,1281.019305,1275.00,84414.487947,1.620665e+06,63.456080,3.789359


In [12]:

predictions_context_512.to_csv("*1. Environment/1. Models/Chronos/Results_bolt/predicitons_512_chronos_bolt.csv")


## L = 256

In [9]:
CONTEXT_LEN = 256

In [10]:
rows_context_256 = []

for i in range(len(overview_data)):
    walk = overview_data.iloc[i]

    context, future = load_window(
        walk["walk_id"], CONTEXT_LEN, ORIGIN, MAX_HORIZON, data_path
    )

    if i % 500 == 0:
        print(f"\r{i}/{len(overview_data)}", end="", flush=True)

    prediction  = forecast_chronos_bolt(context, MAX_HORIZON, pipeline2)
    plugin      = plugin_forecaster(context, MAX_HORIZON)
    conditional = conditional_mean_forecaster(context, MAX_HORIZON, walk["mu"])
    naive       = naive_forecaster(context, MAX_HORIZON)

    for h in HORIZONS:
        rows_context_256.append({
            "walk_id":     walk["walk_id"],
            "snr":         walk["snr"],
            "seed":        walk["seed"],
            "sigma":       walk["sigma"],
            "mu":          walk["mu"],
            "context_len": CONTEXT_LEN,
            "horizon":     h,
            "model":       "chronos-2",
            "checkpoint":  "chronos-2-amazon",
            "num_samples": 100,
            "actual":      future[h - 1],
            "forecast":    prediction[h - 1],
            "naive":       naive[h - 1],
            "plugin":      plugin[h - 1],
            "conditional": conditional[h - 1],
            "se_model":       squared_error(prediction,  future, h),
            "se_naive":       squared_error(naive,       future, h),
            "se_plugin":      squared_error(plugin,      future, h),
            "se_conditional": squared_error(conditional, future, h),
        })

predictions_context_256 = pd.DataFrame(rows_context_256)

12000/12500

In [11]:
predictions_context_256

,walk_id,snr,seed,sigma,mu,context_len,horizon,model,checkpoint,num_samples,actual,forecast,naive,plugin,conditional,se_model,se_naive,se_plugin,se_conditional
0,snr0.01_seed0,0.01,0,1.0,0.01,256,1,chronos-2,chronos-2-amazon,100,0.152914,-0.085079,0.0,-0.040656,0.01,0.056641,2.338258e-02,0.037469,0.020424
1,snr0.01_seed0,0.01,0,1.0,0.01,256,2,chronos-2,chronos-2-amazon,100,-0.381044,-0.075220,0.0,-0.081313,0.02,0.093528,1.451946e-01,0.089839,0.160836
2,snr0.01_seed0,0.01,0,1.0,0.01,256,3,chronos-2,chronos-2-amazon,100,-0.504496,-0.065422,0.0,-0.121969,0.03,0.192786,2.545158e-01,0.146327,0.285686
3,snr0.01_seed0,0.01,0,1.0,0.01,256,4,chronos-2,chronos-2-amazon,100,0.803376,-0.070126,0.0,-0.162625,0.04,0.763007,6.454132e-01,0.933158,0.582743
4,snr0.01_seed0,0.01,0,1.0,0.01,256,5,chronos-2,chronos-2-amazon,100,-0.154444,0.150729,0.0,-0.203281,0.05,0.093130,2.385287e-02,0.002385,0.041797
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3199995,snr5_seed99,5.00,99,1.0,5.00,256,252,chronos-2,chronos-2-amazon,100,1260.902970,1329.069824,0.0,1265.174318,1260.00,4646.720082,1.589876e+06,18.244415,0.815354
3199996,snr5_seed99,5.00,99,1.0,5.00,256,253,chronos-2,chronos-2-amazon,100,1263.939192,1332.835449,0.0,1270.194851,1265.00,4746.694247,1.597542e+06,39.133264,1.125313
3199997,snr5_seed99,5.00,99,1.0,5.00,256,254,chronos-2,chronos-2-amazon,100,1268.951206,1341.268311,0.0,1275.215384,1270.00,5229.763604,1.610237e+06,39.239921,1.099969
3199998,snr5_seed99,5.00,99,1.0,5.00,256,255,chronos-2,chronos-2-amazon,100,1273.053372,1342.806519,0.0,1280.235917,1275.00,4865.501404,1.620665e+06,51.588943,3.789359


In [12]:

predictions_context_256.to_csv("*1. Environment/1. Models/Chronos/Results_bolt/predicitons_256_chronos_bolt.csv")


## L = 128

In [13]:
CONTEXT_LEN = 128

In [14]:
rows_context_128 = []

for i in range(len(overview_data)):
    walk = overview_data.iloc[i]

    context, future = load_window(
        walk["walk_id"], CONTEXT_LEN, ORIGIN, MAX_HORIZON, data_path
    )

    if i % 500 == 0:
        print(f"\r{i}/{len(overview_data)}", end="", flush=True)
        
    prediction  = forecast_chronos_bolt(context, MAX_HORIZON, pipeline2)
    plugin      = plugin_forecaster(context, MAX_HORIZON)
    conditional = conditional_mean_forecaster(context, MAX_HORIZON, walk["mu"])
    naive       = naive_forecaster(context, MAX_HORIZON)

    for h in HORIZONS:
        rows_context_128.append({
            "walk_id":     walk["walk_id"],
            "snr":         walk["snr"],
            "seed":        walk["seed"],
            "sigma":       walk["sigma"],
            "mu":          walk["mu"],
            "context_len": CONTEXT_LEN,
            "horizon":     h,
            "model":       "chronos-2",
            "checkpoint":  "chronos-2-amazon",
            "actual":      future[h - 1],
            "forecast":    prediction[h - 1],
            "naive":       naive[h - 1],
            "plugin":      plugin[h - 1],
            "conditional": conditional[h - 1],
            "se_model":       squared_error(prediction,  future, h),
            "se_naive":       squared_error(naive,       future, h),
            "se_plugin":      squared_error(plugin,      future, h),
            "se_conditional": squared_error(conditional, future, h),
        })

predictions_context_128 = pd.DataFrame(rows_context_128)

12000/12500

In [15]:
predictions_context_128

,walk_id,snr,seed,sigma,mu,context_len,horizon,model,checkpoint,actual,forecast,naive,plugin,conditional,se_model,se_naive,se_plugin,se_conditional
0,snr0.01_seed0,0.01,0,1.0,0.01,128,1,chronos-2,chronos-2-amazon,0.152914,0.078878,0.0,0.011189,0.01,0.005481,2.338258e-02,0.020086,0.020424
1,snr0.01_seed0,0.01,0,1.0,0.01,128,2,chronos-2,chronos-2-amazon,-0.381044,0.095528,0.0,0.022378,0.02,0.227121,1.451946e-01,0.162749,0.160836
2,snr0.01_seed0,0.01,0,1.0,0.01,128,3,chronos-2,chronos-2-amazon,-0.504496,0.096469,0.0,0.033567,0.03,0.361159,2.545158e-01,0.289511,0.285686
3,snr0.01_seed0,0.01,0,1.0,0.01,128,4,chronos-2,chronos-2-amazon,0.803376,0.074780,0.0,0.044756,0.04,0.530852,6.454132e-01,0.575505,0.582743
4,snr0.01_seed0,0.01,0,1.0,0.01,128,5,chronos-2,chronos-2-amazon,-0.154444,0.154738,0.0,0.055945,0.05,0.095593,2.385287e-02,0.044263,0.041797
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3199995,snr5_seed99,5.00,99,1.0,5.00,128,252,chronos-2,chronos-2-amazon,1260.902970,782.672241,0.0,1291.282869,1260.00,228704.629482,1.589876e+06,922.938322,0.815354
3199996,snr5_seed99,5.00,99,1.0,5.00,128,253,chronos-2,chronos-2-amazon,1263.939192,784.362610,0.0,1296.407008,1265.00,229993.698219,1.597542e+06,1054.159060,1.125313
3199997,snr5_seed99,5.00,99,1.0,5.00,128,254,chronos-2,chronos-2-amazon,1268.951206,788.244812,0.0,1301.531146,1270.00,231078.637259,1.610237e+06,1061.452502,1.099969
3199998,snr5_seed99,5.00,99,1.0,5.00,128,255,chronos-2,chronos-2-amazon,1273.053372,784.263611,0.0,1306.655285,1275.00,238915.430964,1.620665e+06,1129.088505,3.789359


In [16]:

predictions_context_128.to_csv("*1. Environment/1. Models/Chronos/Results_bolt/predicitons_128_chronos_bolt.csv")


## L = 64

In [17]:
CONTEXT_LEN = 64

In [18]:
rows_context_64 = []

for i in range(len(overview_data)):
    walk = overview_data.iloc[i]

    context, future = load_window(
        walk["walk_id"], CONTEXT_LEN, ORIGIN, MAX_HORIZON, data_path
    )

    if i % 500 == 0:
        print(f"\r{i}/{len(overview_data)}", end="", flush=True)

    prediction  = forecast_chronos_bolt(context, MAX_HORIZON, pipeline2)
    plugin      = plugin_forecaster(context, MAX_HORIZON)
    conditional = conditional_mean_forecaster(context, MAX_HORIZON, walk["mu"])
    naive       = naive_forecaster(context, MAX_HORIZON)

    for h in HORIZONS:
        rows_context_64.append({
            "walk_id":     walk["walk_id"],
            "snr":         walk["snr"],
            "seed":        walk["seed"],
            "sigma":       walk["sigma"],
            "mu":          walk["mu"],
            "context_len": CONTEXT_LEN,
            "horizon":     h,
            "model":       "chronos-2",
            "checkpoint":  "chronos-2-amazon",
            "actual":      future[h - 1],
            "forecast":    prediction[h - 1],
            "naive":       naive[h - 1],
            "plugin":      plugin[h - 1],
            "conditional": conditional[h - 1],
            "se_model":       squared_error(prediction,  future, h),
            "se_naive":       squared_error(naive,       future, h),
            "se_plugin":      squared_error(plugin,      future, h),
            "se_conditional": squared_error(conditional, future, h),
        })

predictions_context_64 = pd.DataFrame(rows_context_64)

12000/12500

In [19]:
predictions_context_64

,walk_id,snr,seed,sigma,mu,context_len,horizon,model,checkpoint,actual,forecast,naive,plugin,conditional,se_model,se_naive,se_plugin,se_conditional
0,snr0.01_seed0,0.01,0,1.0,0.01,64,1,chronos-2,chronos-2-amazon,0.152914,-0.254611,0.0,-0.102400,0.01,1.660762e-01,2.338258e-02,0.065185,0.020424
1,snr0.01_seed0,0.01,0,1.0,0.01,64,2,chronos-2,chronos-2-amazon,-0.381044,-0.227334,0.0,-0.204800,0.02,2.362678e-02,1.451946e-01,0.031062,0.160836
2,snr0.01_seed0,0.01,0,1.0,0.01,64,3,chronos-2,chronos-2-amazon,-0.504496,-0.211221,0.0,-0.307200,0.03,8.601016e-02,2.545158e-01,0.038926,0.285686
3,snr0.01_seed0,0.01,0,1.0,0.01,64,4,chronos-2,chronos-2-amazon,0.803376,-0.267425,0.0,-0.409600,0.04,1.146616e+00,6.454132e-01,1.471311,0.582743
4,snr0.01_seed0,0.01,0,1.0,0.01,64,5,chronos-2,chronos-2-amazon,-0.154444,-0.261882,0.0,-0.512000,0.05,1.154294e-02,2.385287e-02,0.127846,0.041797
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3199995,snr5_seed99,5.00,99,1.0,5.00,64,252,chronos-2,chronos-2-amazon,1260.902970,152.900024,0.0,1310.411462,1260.00,1.227671e+06,1.589876e+06,2451.090874,0.815354
3199996,snr5_seed99,5.00,99,1.0,5.00,64,253,chronos-2,chronos-2-amazon,1263.939192,151.632324,0.0,1315.611508,1265.00,1.237227e+06,1.597542e+06,2670.028225,1.125313
3199997,snr5_seed99,5.00,99,1.0,5.00,64,254,chronos-2,chronos-2-amazon,1268.951206,150.469818,0.0,1320.811553,1270.00,1.251001e+06,1.610237e+06,2689.495630,1.099969
3199998,snr5_seed99,5.00,99,1.0,5.00,64,255,chronos-2,chronos-2-amazon,1273.053372,146.617584,0.0,1326.011599,1275.00,1.268858e+06,1.620665e+06,2804.573759,3.789359


In [20]:

predictions_context_64.to_csv("*1. Environment/1. Models/Chronos/Results_bolt/predicitons_64_chronos_bolt.csv")


## L = 32

In [21]:
CONTEXT_LEN = 32

In [22]:
rows_context_32 = []

for i in range(len(overview_data)):
    walk = overview_data.iloc[i]

    context, future = load_window(
        walk["walk_id"], CONTEXT_LEN, ORIGIN, MAX_HORIZON, data_path
    )

    if i % 500 == 0:
        print(f"\r{i}/{len(overview_data)}", end="", flush=True)

    prediction  = forecast_chronos_bolt(context, MAX_HORIZON, pipeline2)
    plugin      = plugin_forecaster(context, MAX_HORIZON)
    conditional = conditional_mean_forecaster(context, MAX_HORIZON, walk["mu"])
    naive       = naive_forecaster(context, MAX_HORIZON)

    for h in HORIZONS:
        rows_context_32.append({
            "walk_id":     walk["walk_id"],
            "snr":         walk["snr"],
            "seed":        walk["seed"],
            "sigma":       walk["sigma"],
            "mu":          walk["mu"],
            "context_len": CONTEXT_LEN,
            "horizon":     h,
            "model":       "chronos-2",
            "checkpoint":  "chronos-2-amazon",
            "actual":      future[h - 1],
            "forecast":    prediction[h - 1],
            "naive":       naive[h - 1],
            "plugin":      plugin[h - 1],
            "conditional": conditional[h - 1],
            "se_model":       squared_error(prediction,  future, h),
            "se_naive":       squared_error(naive,       future, h),
            "se_plugin":      squared_error(plugin,      future, h),
            "se_conditional": squared_error(conditional, future, h),
        })

predictions_context_32 = pd.DataFrame(rows_context_32)

0/12500

12000/12500

In [23]:
predictions_context_32

,walk_id,snr,seed,sigma,mu,context_len,horizon,model,checkpoint,actual,forecast,naive,plugin,conditional,se_model,se_naive,se_plugin,se_conditional
0,snr0.01_seed0,0.01,0,1.0,0.01,32,1,chronos-2,chronos-2-amazon,0.152914,-0.092828,0.0,-0.023217,0.01,6.038898e-02,2.338258e-02,0.031022,0.020424
1,snr0.01_seed0,0.01,0,1.0,0.01,32,2,chronos-2,chronos-2-amazon,-0.381044,-0.010955,0.0,-0.046434,0.02,1.369657e-01,1.451946e-01,0.111964,0.160836
2,snr0.01_seed0,0.01,0,1.0,0.01,32,3,chronos-2,chronos-2-amazon,-0.504496,0.062804,0.0,-0.069650,0.03,3.218283e-01,2.545158e-01,0.189090,0.285686
3,snr0.01_seed0,0.01,0,1.0,0.01,32,4,chronos-2,chronos-2-amazon,0.803376,0.036116,0.0,-0.092867,0.04,5.886881e-01,6.454132e-01,0.803252,0.582743
4,snr0.01_seed0,0.01,0,1.0,0.01,32,5,chronos-2,chronos-2-amazon,-0.154444,0.072249,0.0,-0.116084,0.05,5.138938e-02,2.385287e-02,0.001471,0.041797
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3199995,snr5_seed99,5.00,99,1.0,5.00,32,252,chronos-2,chronos-2-amazon,1260.902970,-133.402771,0.0,1309.016155,1260.00,1.944088e+06,1.589876e+06,2314.878649,0.815354
3199996,snr5_seed99,5.00,99,1.0,5.00,32,253,chronos-2,chronos-2-amazon,1263.939192,-133.303177,0.0,1314.210664,1265.00,1.952286e+06,1.597542e+06,2527.220875,1.125313
3199997,snr5_seed99,5.00,99,1.0,5.00,32,254,chronos-2,chronos-2-amazon,1268.951206,-134.369537,0.0,1319.405172,1270.00,1.969309e+06,1.610237e+06,2545.602723,1.099969
3199998,snr5_seed99,5.00,99,1.0,5.00,32,255,chronos-2,chronos-2-amazon,1273.053372,-135.376465,0.0,1324.599681,1275.00,1.983675e+06,1.620665e+06,2657.021932,3.789359


In [24]:

predictions_context_32.to_csv("*1. Environment/1. Models/Chronos/Results_bolt/predicitons_32_chronos_bolt.csv")
